# Module 7 • Hugging Face and Pretrained Transformer Workflows

# Lesson 37 • Token Classification with Pretrained Transformers

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Execution target:** CPU by default

---

## Scope

This lesson develops token classification with pretrained Transformers, focusing on
named-entity recognition (NER). It covers BIO tagging, word-to-subword alignment,
ignored labels, dynamic padding, padding-aware loss, span-level evaluation, error
analysis, checkpointing, and optional Hugging Face workflows.

The executable core runs offline on CPU. Optional Hugging Face cells remain disabled
unless the learner explicitly enables them.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain token classification and NER;
- create and validate BIO labels;
- align word labels with subword tokens;
- use `-100` for special tokens, continuation pieces, and padding;
- train a Transformer token classifier;
- evaluate token-level and span-level performance;
- diagnose boundary, type, missed, and spurious entity errors;
- structure AutoTokenizer, AutoModelForTokenClassification, pipeline, collator, and
  Trainer workflows;
- assess Arabic morphology, clitics, and tashkeel in token classification.

## Table of Contents

1. Token Classification and NER
2. BIO Tagging
3. BIO Validation
4. Subword Alignment
5. Ignore Index and Loss
6. Offline NER Corpus
7. Data Splits
8. Label and Word Vocabularies
9. Dataset and Dynamic Padding
10. Transformer Token Classifier
11. Training
12. Learning Curves
13. Token-Level Evaluation
14. Span Extraction
15. Span-Level Evaluation
16. Qualitative Analysis
17. Error Taxonomy
18. Confidence Analysis
19. Saving and Reloading
20. Simulated Subword Alignment
21. Optional Hugging Face Setup
22. Optional Tokenizer Alignment
23. Optional AutoModel and Pipeline
24. Optional Trainer and Collator
25. Long Sequences and Imbalance
26. Arabic and Multilingual Considerations
27. Reproducibility
28. Knowledge Check
29. Exercises
30. Summary and Next Lesson

# 1. Token Classification and NER

Token classification assigns one label to every token. Common applications include
named-entity recognition, part-of-speech tagging, chunking, and slot filling.

In [ ]:
import copy
import importlib.util
import json
import math
import platform
import random
import tempfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

tasks = pd.DataFrame(
    [
        ("Named-entity recognition", "entity type per token"),
        ("Part-of-speech tagging", "grammatical category"),
        ("Chunking", "phrase boundary"),
        ("Slot filling", "intent argument"),
    ],
    columns=["Task", "Output"],
)
tasks

NER identifies spans such as persons, organizations, locations, dates, and products.

# 2. BIO Tagging

- `B-X`: beginning of an entity of type X;
- `I-X`: continuation of an entity of type X;
- `O`: outside an entity.

In [ ]:
pd.DataFrame(
    {
        "token": ["Alice", "joined", "OpenAI", "in", "Cairo"],
        "label": ["B-PER", "O", "B-ORG", "O", "B-LOC"],
    }
)

# 3. BIO Validation

An `I-X` label should follow `B-X` or `I-X`.

In [ ]:
def validate_bio(labels):
    errors = []
    previous_prefix = "O"
    previous_type = None

    for index, label in enumerate(labels):
        if label == "O":
            previous_prefix = "O"
            previous_type = None
            continue

        prefix, entity_type = label.split("-", 1)

        if prefix == "I" and (
            previous_prefix not in {"B", "I"} or previous_type != entity_type
        ):
            errors.append(f"Illegal transition at {index}: {label}")

        previous_prefix = prefix
        previous_type = entity_type

    return errors

validate_bio(["B-PER", "O", "I-ORG"])

# 4. Subword Alignment

Pretrained tokenizers may split one word into several pieces. Common alignment
policies are:

- label only the first piece and ignore later pieces;
- copy the word label to every piece;
- convert later pieces from `B-X` to `I-X`.

The policy must be documented.

In [ ]:
alignment_policies = pd.DataFrame(
    [
        ("First piece only", "continuations use -100"),
        ("Replicate label", "same label on every piece"),
        ("BIO continuation", "B-X then I-X"),
    ],
    columns=["Policy", "Continuation treatment"],
)
alignment_policies

# 5. Ignore Index and Loss

Special tokens and padding do not correspond to original word labels. PyTorch
cross-entropy commonly ignores targets equal to `-100`.

In [ ]:
IGNORE_INDEX = -100

loss_demo = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
logits_demo = torch.tensor([[[2.0, 0.1], [0.2, 1.8], [1.0, 1.0]]])
labels_demo = torch.tensor([[0, 1, IGNORE_INDEX]])

float(loss_demo(logits_demo.reshape(-1, 2), labels_demo.reshape(-1)))

# 6. Offline NER Corpus

The synthetic corpus contains PER, ORG, and LOC entities, including multiword
organizations and locations.

In [ ]:
persons = ["Alice", "Bob", "Carol", "David", "Eman", "Farah"]
organizations = [
    ["OpenAI"],
    ["Google"],
    ["Microsoft"],
    ["Ain", "Shams", "University"],
    ["Cairo", "University"],
    ["Nile", "Research", "Lab"],
]
locations = [
    ["Cairo"],
    ["Alexandria"],
    ["Boston"],
    ["New", "York"],
    ["London"],
    ["Dubai"],
]

templates = [
    (["{PER}", "joined", "{ORG}", "in", "{LOC}"], ["PER", None, "ORG", None, "LOC"]),
    (["{ORG}", "hired", "{PER}", "from", "{LOC}"], ["ORG", None, "PER", None, "LOC"]),
    (["{PER}", "visited", "{LOC}", "with", "{ORG}"], ["PER", None, "LOC", None, "ORG"]),
    (["{ORG}", "sent", "{PER}", "to", "{LOC}"], ["ORG", None, "PER", None, "LOC"]),
    (["in", "{LOC}", "{PER}", "met", "{ORG}"], [None, "LOC", "PER", None, "ORG"]),
]

def append_entity(tokens, labels, entity_tokens, entity_type):
    for index, token in enumerate(entity_tokens):
        tokens.append(token)
        labels.append(f"{'B' if index == 0 else 'I'}-{entity_type}")

examples = []
for template_index, (template_tokens, template_types) in enumerate(templates):
    for index in range(18):
        values = {
            "PER": [persons[(index + template_index) % len(persons)]],
            "ORG": organizations[(2 * index + template_index) % len(organizations)],
            "LOC": locations[(3 * index + template_index) % len(locations)],
        }

        tokens, labels = [], []
        for token_pattern, entity_type in zip(template_tokens, template_types):
            if entity_type is None:
                tokens.append(token_pattern)
                labels.append("O")
            else:
                append_entity(tokens, labels, values[entity_type], entity_type)

        examples.append({"tokens": tokens, "labels": labels})

print("Examples:", len(examples))
examples[0]

# 7. Data Splits

The split is deterministic. Training updates parameters, validation selects the
checkpoint, and test data is reserved for final evaluation.

In [ ]:
random.seed(42)
random.shuffle(examples)

train_end = int(0.70 * len(examples))
validation_end = int(0.85 * len(examples))

train_examples = examples[:train_end]
validation_examples = examples[train_end:validation_end]
test_examples = examples[validation_end:]

pd.Series(
    {
        "training": len(train_examples),
        "validation": len(validation_examples),
        "test": len(test_examples),
    }
)

# 8. Label and Word Vocabularies

In [ ]:
label_names = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
label2id = {label: index for index, label in enumerate(label_names)}
id2label = {index: label for label, index in label2id.items()}

word_counts = Counter(
    token.lower()
    for example in train_examples
    for token in example["tokens"]
)

vocabulary = ["<PAD>", "<UNK>"] + sorted(word_counts)
token2id = {token: index for index, token in enumerate(vocabulary)}

PAD_ID = token2id["<PAD>"]
UNK_ID = token2id["<UNK>"]

print("Labels:", label2id)
print("Vocabulary size:", len(vocabulary))

# 9. Dataset and Dynamic Padding

In [ ]:
class NERDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        return {
            "input_ids": torch.tensor(
                [token2id.get(token.lower(), UNK_ID) for token in example["tokens"]],
                dtype=torch.long,
            ),
            "labels": torch.tensor(
                [label2id[label] for label in example["labels"]],
                dtype=torch.long,
            ),
            "tokens": example["tokens"],
        }

def collate_ner(batch):
    max_length = max(len(item["input_ids"]) for item in batch)

    input_ids = torch.full((len(batch), max_length), PAD_ID, dtype=torch.long)
    labels = torch.full((len(batch), max_length), IGNORE_INDEX, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_length), dtype=torch.long)

    for row, item in enumerate(batch):
        length = len(item["input_ids"])
        input_ids[row, :length] = item["input_ids"]
        labels[row, :length] = item["labels"]
        attention_mask[row, :length] = 1

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
        "padding_mask": attention_mask == 0,
        "tokens": [item["tokens"] for item in batch],
    }

train_loader = DataLoader(
    NERDataset(train_examples),
    batch_size=8,
    shuffle=True,
    collate_fn=collate_ner,
    generator=torch.Generator().manual_seed(42),
)
validation_loader = DataLoader(
    NERDataset(validation_examples),
    batch_size=8,
    shuffle=False,
    collate_fn=collate_ner,
)
test_loader = DataLoader(
    NERDataset(test_examples),
    batch_size=8,
    shuffle=False,
    collate_fn=collate_ner,
)

sample_batch = next(iter(train_loader))
print(sample_batch["input_ids"].shape, sample_batch["labels"].shape)

# 10. Transformer Token Classifier

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, model_dimension, maximum_length=128):
        super().__init__()
        encoding = torch.zeros(maximum_length, model_dimension)
        positions = torch.arange(maximum_length, dtype=torch.float32).unsqueeze(1)
        rates = torch.exp(
            torch.arange(0, model_dimension, 2, dtype=torch.float32)
            * (-math.log(10000.0) / model_dimension)
        )
        encoding[:, 0::2] = torch.sin(positions * rates)
        encoding[:, 1::2] = torch.cos(positions * rates)
        self.register_buffer("encoding", encoding.unsqueeze(0))

    def forward(self, embeddings):
        return embeddings + self.encoding[:, : embeddings.size(1), :]

class TransformerTokenClassifier(nn.Module):
    def __init__(
        self,
        vocabulary_size,
        label_count,
        model_dimension=40,
        head_count=4,
        layer_count=2,
        feed_forward_dimension=80,
        dropout=0.10,
    ):
        super().__init__()
        self.model_dimension = model_dimension
        self.embedding = nn.Embedding(
            vocabulary_size,
            model_dimension,
            padding_idx=PAD_ID,
        )
        self.position = PositionalEncoding(model_dimension)

        layer = nn.TransformerEncoderLayer(
            d_model=model_dimension,
            nhead=head_count,
            dim_feedforward=feed_forward_dimension,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layer_count)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(model_dimension, label_count)

    def forward(self, input_ids, padding_mask):
        embeddings = self.embedding(input_ids) * math.sqrt(self.model_dimension)
        hidden = self.encoder(
            self.position(embeddings),
            src_key_padding_mask=padding_mask,
        )
        logits = self.classifier(self.dropout(hidden))
        return {"logits": logits, "last_hidden_state": hidden}

DEVICE = torch.device("cpu")
torch.manual_seed(42)

model = TransformerTokenClassifier(
    vocabulary_size=len(vocabulary),
    label_count=len(label_names),
).to(DEVICE)

print(
    "Trainable parameters:",
    sum(parameter.numel() for parameter in model.parameters()),
)

The output shape is `(batch_size, sequence_length, number_of_labels)`.

In [ ]:
with torch.no_grad():
    output = model(
        sample_batch["input_ids"].to(DEVICE),
        sample_batch["padding_mask"].to(DEVICE),
    )

print("Logits:", output["logits"].shape)
print("Hidden states:", output["last_hidden_state"].shape)

# 11. Training

In [ ]:
loss_function = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def evaluate_model(model, loader):
    model.eval()
    losses = []
    all_labels = []
    all_predictions = []
    all_probabilities = []
    records = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            padding_mask = batch["padding_mask"].to(DEVICE)

            output = model(input_ids, padding_mask)
            logits = output["logits"]

            loss = loss_function(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
            )

            probabilities = torch.softmax(logits, dim=-1)
            predictions = probabilities.argmax(dim=-1)
            valid_mask = labels != IGNORE_INDEX

            all_labels.extend(labels[valid_mask].cpu().tolist())
            all_predictions.extend(predictions[valid_mask].cpu().tolist())
            all_probabilities.extend(probabilities[valid_mask].cpu().tolist())
            losses.append(float(loss.item()))

            for row, tokens in enumerate(batch["tokens"]):
                length = len(tokens)
                records.append(
                    {
                        "tokens": tokens,
                        "gold": [
                            id2label[int(value)]
                            for value in labels[row, :length].cpu().tolist()
                        ],
                        "predicted": [
                            id2label[int(value)]
                            for value in predictions[row, :length].cpu().tolist()
                        ],
                        "confidence": [
                            float(value)
                            for value in probabilities[row, :length].max(dim=-1).values.cpu().tolist()
                        ],
                    }
                )

    labels_array = np.asarray(all_labels)
    predictions_array = np.asarray(all_predictions)

    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(labels_array, predictions_array),
        "macro_f1": f1_score(
            labels_array,
            predictions_array,
            labels=list(range(len(label_names))),
            average="macro",
            zero_division=0,
        ),
        "labels": labels_array,
        "predictions": predictions_array,
        "probabilities": np.asarray(all_probabilities),
        "records": records,
    }

def train_model(model, epochs=40, learning_rate=0.003, patience=8):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4,
    )

    best_state = copy.deepcopy(model.state_dict())
    best_f1 = -1.0
    no_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        losses = []
        gradient_norms = []

        for batch in train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            padding_mask = batch["padding_mask"].to(DEVICE)

            optimizer.zero_grad()
            output = model(input_ids, padding_mask)
            logits = output["logits"]

            loss = loss_function(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
            )
            loss.backward()
            gradient_norm = clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            losses.append(float(loss.item()))
            gradient_norms.append(float(gradient_norm))

        validation = evaluate_model(model, validation_loader)

        history.append(
            {
                "epoch": epoch,
                "training_loss": float(np.mean(losses)),
                "validation_loss": validation["loss"],
                "validation_accuracy": validation["accuracy"],
                "validation_macro_f1": validation["macro_f1"],
                "gradient_norm": float(np.mean(gradient_norms)),
            }
        )

        if validation["macro_f1"] > best_f1 + 1e-6:
            best_f1 = validation["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())
            no_improvement = 0
        else:
            no_improvement += 1

        if no_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

set_seed(42)
trained_model, training_history = train_model(model)

print("Epochs completed:", len(training_history))
print("Best validation macro F1:", round(training_history["validation_macro_f1"].max(), 3))

# 12. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(training_history["epoch"], training_history["training_loss"], label="Training")
plt.plot(training_history["epoch"], training_history["validation_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Token Classification Learning Curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(training_history["epoch"], training_history["validation_macro_f1"])
plt.xlabel("Epoch")
plt.ylabel("Validation macro F1")
plt.title("Validation Token-Level Macro F1")
plt.tight_layout()
plt.show()

# 13. Token-Level Evaluation

Token accuracy may be inflated by the frequent `O` class, so macro F1 and per-label
metrics should also be reported.

In [ ]:
test_metrics = evaluate_model(trained_model, test_loader)

print("Token accuracy:", round(test_metrics["accuracy"], 3))
print("Token macro F1:", round(test_metrics["macro_f1"], 3))
print()
print(
    classification_report(
        test_metrics["labels"],
        test_metrics["predictions"],
        labels=list(range(len(label_names))),
        target_names=label_names,
        zero_division=0,
    )
)

In [ ]:
matrix = confusion_matrix(
    test_metrics["labels"],
    test_metrics["predictions"],
    labels=list(range(len(label_names))),
)

pd.DataFrame(
    matrix,
    index=[f"actual_{label}" for label in label_names],
    columns=[f"predicted_{label}" for label in label_names],
)

# 14. Span Extraction

A BIO span is represented as `(start, end_exclusive, entity_type)`.

In [ ]:
def extract_bio_spans(labels):
    spans = set()
    start = None
    entity_type = None

    for index, label in enumerate(labels + ["O"]):
        if label == "O":
            prefix = "O"
            current_type = None
        else:
            prefix, current_type = label.split("-", 1)

        should_close = (
            start is not None
            and (
                prefix in {"O", "B"}
                or current_type != entity_type
            )
        )

        if should_close:
            spans.add((start, index, entity_type))
            start = None
            entity_type = None

        if prefix == "B":
            start = index
            entity_type = current_type
        elif prefix == "I" and start is None:
            start = index
            entity_type = current_type

    return spans

extract_bio_spans(["B-PER", "O", "B-ORG", "I-ORG", "O"])

# 15. Span-Level Evaluation

A correct entity requires the correct start, end, and type.

In [ ]:
def span_metrics(records):
    true_positive = 0
    false_positive = 0
    false_negative = 0

    for record in records:
        gold = extract_bio_spans(record["gold"])
        predicted = extract_bio_spans(record["predicted"])

        true_positive += len(gold & predicted)
        false_positive += len(predicted - gold)
        false_negative += len(gold - predicted)

    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    f1_value = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1_value,
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
    }

pd.Series(span_metrics(test_metrics["records"]))

# 16. Qualitative Analysis

In [ ]:
rows = []
for sentence_index, record in enumerate(test_metrics["records"]):
    for token, gold, predicted, confidence in zip(
        record["tokens"],
        record["gold"],
        record["predicted"],
        record["confidence"],
    ):
        rows.append(
            {
                "sentence": sentence_index,
                "token": token,
                "gold": gold,
                "predicted": predicted,
                "confidence": confidence,
                "correct": gold == predicted,
            }
        )

qualitative_frame = pd.DataFrame(rows)
qualitative_frame.head(30)

# 17. Error Taxonomy

Typical errors include:

- missed entity;
- spurious entity;
- boundary error;
- type error;
- illegal BIO transition.

In [ ]:
def categorize_span_errors(gold_spans, predicted_spans):
    categories = []

    for predicted in predicted_spans - gold_spans:
        p_start, p_end, p_type = predicted
        overlaps = [
            gold
            for gold in gold_spans
            if not (p_end <= gold[0] or p_start >= gold[1])
        ]

        if not overlaps:
            categories.append("spurious entity")
        elif any(
            p_start == gold[0] and p_end == gold[1] and p_type != gold[2]
            for gold in overlaps
        ):
            categories.append("type error")
        else:
            categories.append("boundary error")

    for gold in gold_spans - predicted_spans:
        if not any(
            not (predicted[1] <= gold[0] or predicted[0] >= gold[1])
            for predicted in predicted_spans
        ):
            categories.append("missed entity")

    return categories

error_categories = []
for record in test_metrics["records"]:
    error_categories.extend(
        categorize_span_errors(
            extract_bio_spans(record["gold"]),
            extract_bio_spans(record["predicted"]),
        )
    )

pd.Series(error_categories, dtype="object").value_counts()

# 18. Confidence Analysis

High confidence does not guarantee correctness.

In [ ]:
qualitative_frame.groupby("correct")["confidence"].describe()

# 19. Saving and Reloading

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = Path(directory) / "token_classifier.pt"

    torch.save(
        {
            "model_state_dict": trained_model.state_dict(),
            "vocabulary": vocabulary,
            "label2id": label2id,
            "id2label": id2label,
        },
        checkpoint_path,
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    reloaded_model = TransformerTokenClassifier(
        vocabulary_size=len(checkpoint["vocabulary"]),
        label_count=len(checkpoint["label2id"]),
    ).to(DEVICE)

    reloaded_model.load_state_dict(checkpoint["model_state_dict"])
    reload_metrics = evaluate_model(reloaded_model, test_loader)

print("Reloaded accuracy:", round(reload_metrics["accuracy"], 3))

# 20. Simulated Subword Alignment

In [ ]:
def toy_wordpiece_split(word):
    lowered = word.lower()
    if len(lowered) <= 6:
        return [lowered]
    return [lowered[:4], "##" + lowered[4:]]

def align_first_subword(words, labels):
    subwords = ["[CLS]"]
    word_ids = [None]
    aligned_labels = [IGNORE_INDEX]

    for word_index, (word, label) in enumerate(zip(words, labels)):
        pieces = toy_wordpiece_split(word)

        for piece_index, piece in enumerate(pieces):
            subwords.append(piece)
            word_ids.append(word_index)
            aligned_labels.append(
                label2id[label] if piece_index == 0 else IGNORE_INDEX
            )

    subwords.append("[SEP]")
    word_ids.append(None)
    aligned_labels.append(IGNORE_INDEX)

    return pd.DataFrame(
        {
            "subword": subwords,
            "word_id": word_ids,
            "aligned_label": aligned_labels,
        }
    )

align_first_subword(
    ["Alice", "visited", "Alexandria"],
    ["B-PER", "O", "B-LOC"],
)

# 21. Optional Hugging Face Setup

The following demonstrations remain disabled by default.

In [ ]:
TRANSFORMERS_AVAILABLE = (
    importlib.util.find_spec("transformers") is not None
)
DATASETS_AVAILABLE = (
    importlib.util.find_spec("datasets") is not None
)

RUN_HUGGING_FACE_DEMOS = False
USE_LOCAL_FILES_ONLY = True
MODEL_ID = "distilbert/distilbert-base-uncased"

pd.Series(
    {
        "transformers installed": TRANSFORMERS_AVAILABLE,
        "datasets installed": DATASETS_AVAILABLE,
        "run demos": RUN_HUGGING_FACE_DEMOS,
        "local files only": USE_LOCAL_FILES_ONLY,
        "model ID": MODEL_ID,
    }
)

# 22. Optional Tokenizer Alignment

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        use_fast=True,
        local_files_only=USE_LOCAL_FILES_ONLY,
    )

    words = ["Alice", "visited", "Alexandria"]
    labels = ["B-PER", "O", "B-LOC"]

    encoded = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        return_tensors="pt",
    )

    word_ids = encoded.word_ids(batch_index=0)
    aligned = []
    previous_word_id = None

    for word_id in word_ids:
        if word_id is None:
            aligned.append(IGNORE_INDEX)
        elif word_id != previous_word_id:
            aligned.append(label2id[labels[word_id]])
        else:
            aligned.append(IGNORE_INDEX)
        previous_word_id = word_id

    print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))
    print(aligned)
else:
    print("Optional tokenizer alignment skipped.")

# 23. Optional AutoModel and Pipeline

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import AutoModelForTokenClassification

    hf_model = AutoModelForTokenClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(label_names),
        label2id=label2id,
        id2label=id2label,
        local_files_only=USE_LOCAL_FILES_ONLY,
    ).to("cpu")

    with torch.no_grad():
        hf_output = hf_model(**encoded)

    print("Logits:", hf_output.logits.shape)
else:
    print("Optional AutoModel demonstration skipped.")

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import pipeline

    ner_pipeline = pipeline(
        "token-classification",
        model=MODEL_ID,
        tokenizer=MODEL_ID,
        aggregation_strategy="simple",
        device=-1,
        model_kwargs={"local_files_only": USE_LOCAL_FILES_ONLY},
    )

    print(ner_pipeline("Alice visited Alexandria."))
else:
    print("Optional token-classification pipeline skipped.")

# 24. Optional Trainer and Collator

`DataCollatorForTokenClassification` dynamically pads model inputs and token labels.
Its label padding value is commonly `-100`.

In [ ]:
trainer_template = '''
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

arguments = TrainingArguments(
    output_dir="checkpoints/ner",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=arguments,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
'''

print(trainer_template)

Trainer argument names can change across library versions, so the template should be
checked against the installed documentation.

# 25. Long Sequences and Imbalance

NER datasets contain many `O` tokens. Report entity-level metrics rather than relying
only on token accuracy.

Documents longer than the model context can be processed with overlapping windows,
sentence segmentation, or document-level architectures. Entity spans must be
reconciled across windows.

In [ ]:
label_distribution = Counter(
    label
    for example in train_examples
    for label in example["labels"]
)

pd.Series(label_distribution).sort_values(ascending=False)

# 26. Arabic and Multilingual Considerations

Arabic token classification is affected by:

- attached clitics;
- rich morphology;
- optional tashkeel;
- orthographic variation;
- tokenizer fragmentation;
- MSA and dialect differences;
- multiword entities.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "وَزَارَتْ إِيمَانُ الْقَاهِرَةَ",
            ["O", "B-PER", "B-LOC"],
        ),
        (
            "عَمِلَتْ فِي جَامِعَةِ عَيْنِ شَمْسٍ",
            ["O", "O", "B-ORG", "I-ORG", "I-ORG"],
        ),
    ],
    columns=["Fully vocalized sentence", "Illustrative BIO labels"],
)

arabic_examples

For fully vocalized Arabic tasks, tashkeel must remain in tokenization, alignment,
evaluation, and deployment whenever it is part of the task definition.

Clitic segmentation changes token boundaries and therefore requires label realignment.

In [ ]:
pd.DataFrame(
    [
        ("Tashkeel", "preserve consistently"),
        ("Clitics", "realign labels after segmentation"),
        ("Tokenizer", "measure subword fragmentation"),
        ("Language variety", "separate MSA and dialect analysis"),
        ("Entities", "inspect multiword boundaries"),
    ],
    columns=["Check", "Action"],
)

# 27. Reproducibility

Report:

- corpus and annotation scheme;
- entity types;
- train, validation, and test splits;
- tokenizer and revision;
- subword alignment policy;
- ignore index;
- maximum sequence length;
- padding strategy;
- label mappings;
- learning rate and batch size;
- checkpoint-selection metric;
- token- and span-level metrics;
- random seed, software versions, and hardware.

In [ ]:
reproducibility_metadata = pd.Series(
    {
        "training examples": len(train_examples),
        "validation examples": len(validation_examples),
        "test examples": len(test_examples),
        "labels": len(label_names),
        "ignore index": IGNORE_INDEX,
        "device": str(DEVICE),
        "seed": 42,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers installed": TRANSFORMERS_AVAILABLE,
        "optional demos enabled": RUN_HUGGING_FACE_DEMOS,
    },
    name="Lesson 37 experiment",
)

reproducibility_metadata

# 28. Knowledge Check

1. What is token classification?
2. What do B, I, and O mean?
3. Why can an `I-ORG` transition be illegal?
4. Why must word labels be aligned with subwords?
5. Why are special-token labels set to `-100`?
6. Why can token accuracy be misleading?
7. What does span-level F1 require?
8. What is a boundary error?
9. What is a type error?
10. What does `DataCollatorForTokenClassification` do?
11. How can long documents be handled?
12. Why do Arabic clitics complicate NER?
13. Why must tashkeel policy remain consistent?

# 29. Exercises

1. Add DATE and PRODUCT entity types.
2. Compare first-subword and replicated-label alignment.
3. Calculate per-entity-type span F1.
4. Add weighted token loss.
5. Implement legal BIO transition decoding.
6. Process a long document with overlapping windows.
7. Fine-tune `AutoModelForTokenClassification`.
8. Compare raw token predictions with pipeline aggregation.
9. Build a fully vocalized MSA NER dataset.
10. Produce a boundary, type, missed, and spurious entity report.

## Challenge Exercises

- add a CRF layer;
- support nested entities;
- compare multilingual and Arabic-specific encoders;
- publish a model card with span-level metrics and limitations.

# 30. Summary and Next Lesson

In this lesson:

- token classification and NER were defined;
- BIO labels and legal transitions were examined;
- word labels were aligned with subword pieces;
- ignored labels protected special tokens, continuation pieces, and padding;
- a complete CPU-only Transformer token classifier was trained;
- token accuracy, macro F1, and confusion matrices were calculated;
- BIO spans were extracted and evaluated with span-level precision, recall, and F1;
- boundary, type, missed, and spurious entity errors were analyzed;
- checkpoints preserved weights, vocabulary, and label mappings;
- optional Hugging Face tokenizer, AutoModel, pipeline, collator, and Trainer
  workflows were provided;
- Arabic morphology, clitics, multilingual variation, and tashkeel were integrated.

## Next Lesson

**Lesson 38: Question Answering with Pretrained Transformers** introduces extractive
question answering, context-question tokenization, answer-span alignment, start and
end logits, overflow windows, no-answer handling, and evaluation.

# References

- Hugging Face Transformers documentation: token classification, fast tokenizers,
  word IDs, data collators, pipelines, and Trainer.
- Devlin, J. et al. BERT.
- Sang, E. F. T. K., & De Meulder, F. CoNLL named-entity recognition.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.